<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 23b (əlavə)</h2>
<h2>Rəqəmsal İmzalar və DSA</h2>
<h3 style="color: #8B4513;">Asimmetrik autentifikasiya və açıq doğrulama</h3>
<p><b>Ön şərt:</b> RSA, sonlu sahələr və elliptik əyrilər; bu əlavə praktika Mühazirə 23-dən sonra istifadə olunur.</p>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [Lazım olan kitabxanalar](#lazım-olan-kitabxanalar)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: Rəqəmsal imza nədir? (10 dəq)](#xatırlatma-rəqəmsal-imza-nədir-10-dəq)
5. [RSA imza sxemi (20 dəq)](#rsa-imza-sxemi-20-dəq)
6. [DSA (Digital Signature Algorithm) (25 dəq)](#dsa-digital-signature-algorithm-25-dəq)
7. [ECDSA (Elliptic Curve DSA) (20 dəq)](#ecdsa-elliptic-curve-dsa-20-dəq)
8. [Ed25519 - Müasir imza sxemi (15 dəq)](#ed25519-müasir-imza-sxemi-15-dəq)
9. [Rəqəmsal imzaların müqayisəsi (10 dəq)](#rəqəmsal-imzaların-müqayisəsi-10-dəq)
10. [İnteqrasiya edilmiş tətbiq (15 dəq)](#inteqrasiya-edilmiş-tətbiq-15-dəq)
11. [Ev tapşırığı](#ev-tapşırığı)
12. [Yekun və müzakirə sualları](#yekun-və-müzakirə-sualları)
13. [Əlavə resurslar](#əlavə-resurslar)

## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ Rəqəmsal imzanın konsepsiyasını, MAK-dan fərqini və inkar-edilməzliyin protokol/hüquqi kontekstdən asılı olduğunu izah edə biləcəksiniz  
✅ RSA imza sxeminin iş prinsipini başa düşəcək və implementasiya edə biləcəksiniz  
✅ DSA (Digital Signature Algorithm) alqoritminin riyazi əsaslarını və iş prinsipini izah edə biləcəksiniz  
✅ ECDSA (Elliptic Curve DSA) və Ed25519 kimi müasir imza sxemlərini başa düşəcəksiniz  
✅ Python-da müxtəlif imza sxemlərini (cryptography, PyCryptodome) istifadə edə biləcəksiniz  
✅ Rəqəmsal imzaların praktik tətbiqlərini (SSL sertifikatları, kriptovalyutalar, kod imzalaması) biləcəksiniz

## 📚 Lazım olan kitabxanalar

| Kitabxana | Quraşdırma | İstifadə sahəsi |
|-----------|------------|-----------------|
| cryptography | `!pip install cryptography` | RSA, DSA, ECDSA, Ed25519 |
| PyCryptodome | `!pip install pycryptodome` | RSA, DSA, PKCS#1 v1.5 |
| hashlib | Python standart kitabxanası | Heş funksiyaları |

> 💡 **Qeyd:** `cryptography` kitabxanası müasir imza sxemləri üçün ən yaxşı seçimdir. PyCryptodome isə klassik RSA/DSA implementasiyaları üçün istifadə olunur.

In [ ]:
# Lazımi kitabxanaların quraşdırılması

import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "cryptography", "pycryptodome", "--quiet"],
    check=True,
)
print("✅ Kitabxanalar quraşdırıldı")

## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı kodu işə salaraq lazımi modulların yükləndiyini yoxlayın:

In [ ]:
import sys
import hashlib
import os
import time
import math

from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa, dsa, ec, utils
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.exceptions import InvalidSignature
from Crypto.PublicKey import RSA, DSA
from Crypto.Signature import pkcs1_15, DSS
from Crypto.Hash import SHA256
from Crypto.Random import get_random_bytes

CRYPTOGRAPHY_AVAILABLE = True
CRYPTO_AVAILABLE = True
print("✅ cryptography və PyCryptodome yüklü; bütün imza demoları icra ediləcək")

print(f"\n🐍 Python versiyası: {sys.version}")

### 3.2 Fayl sistemi müqaviləsi

Notebook cari işçi qovluğu dəyişmir və fayl yaratmır. Bütün nümunələr yaddaşda işləyir; beləliklə ardıcıl clean-run başqa praktikaların path davranışına təsir etmir.


In [ ]:
RUN_CONTRACT = {
    "filesystem_writes": 0,
    "working_directory_changes": 0,
    "interactive_prompts_executed": 0,
}
assert all(value == 0 for value in RUN_CONTRACT.values())
print("✅ AS-23b fayl sistemi müqaviləsi təsdiqləndi:", RUN_CONTRACT)


## 📖 Xatırlatma: Rəqəmsal imza nədir? (10 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 Rəqəmsal imza nədir?</h4>
<p>Rəqəmsal imza — elektron sənədin həqiqiliyini, tamlığını və mənbəsini təsdiqləyən kriptoqrafik mexanizmdir. MAK-dan fərqli olaraq, asimmetrik kriptoqrafiyadan istifadə edir.</p>

<p style="text-align: center; font-size: 1.2em;">$$\text{İmza} = \text{Sign}_{sk}(M)$$</p>
<p style="text-align: center; font-size: 1.2em;">$$\text{Doğrulama} = \text{Verify}_{pk}(M, \text{İmza})$$</p>

<p>burada:</p>
<ul>
    <li>$sk$ — gizli imza açarı (private key)</li>
    <li>$pk$ — açıq doğrulama açarı (public key)</li>
    <li>$M$ — mesaj</li>
</ul>
</div>

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50; margin-top: 10px;">
<h4>📌 Rəqəmsal imza istifadəsi</h4>
<p>Alis Boba mesaj göndərir:</p>
<ol>
    <li>Alis: $s = \text{Sign}_{sk}(M)$ hesablayır (gizli açarı ilə)</li>
    <li>Alis Boba $(M, s)$ göndərir; Bob $pk$ açarını autentifikasiya olunmuş mənbədən (məsələn, sertifikat və ya əvvəlcədən yoxlanmış fingerprint) alır</li>
    <li>Bob: $\text{Verify}_{pk}(M, s)$ yoxlayır</li>
</ol>
<p><b>İnkar-edilməzlik:</b> imzanın açıq yoxlanması texniki sübut yaradır, lakin nəticə açarın sahibə düzgün bağlanmasından, gizli açarın həyat dövründən, auditdən və hüquqi/protokol kontekstindən asılıdır.</p>
</div>

<div style="background-color: #f0f0f0; padding: 15px; border-radius: 10px; border-left: 5px solid #808080; margin-top: 10px;">
<h4>⚖️ MAK vs Rəqəmsal imza</h4>
<table>
<tr><th>Xüsusiyyət</th><th>MAK</th><th>Rəqəmsal imza</th></tr>
<tr><td>Açar növü</td><td>Simmetrik</td><td>Asimmetrik</td></tr>
<tr><td>Tərəf sayı</td><td>2 tərəf</td><td>Çox tərəf</td></tr>
<tr><td>Açıq yoxlama</td><td>❌ Xeyr: tərəflər ortaq sirri bilir</td><td>✅ Bəli: doğrulama açarı açıqdır</td></tr>
<tr><td>Sürət</td><td>Adətən daha sürətli</td><td>Alqoritm/əməliyyat/platformadan asılıdır</td></tr>
<tr><td>Açar paylanması</td><td>Ortaq sirr əvvəlcədən lazımdır</td><td>Açıq açar autentifikasiya edilməlidir</td></tr>
</table>
</div>

## 🔐 RSA imza sxemi (20 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔐 RSA imza sxemi</h4>
<p>RSA imza və RSA şifrələmə eyni RSA trapdoor permutasiyasından istifadə etsə də, imza "şifrələmənin tərsi" deyil: fərqli açar istifadəsi və fərqli encoding/padding sxemləri var.</p>

<p><b>Açar generasiyası:</b></p>
<ul>
    <li>$n = p \cdot q$ (iki böyük sadə ədədin hasili)</li>
    <li>$\phi(n) = (p-1)(q-1)$</li>
    <li>$e$ — açıq açar (adətən 65537)</li>
    <li>$d \equiv e^{-1} \pmod{\phi(n)}$ — gizli açar</li>
</ul>

<p><b>Textbook model:</b> kodlanmamış nümayəndə üçün $s = m^d \mod n$ və $m' = s^e \mod n$ yazıla bilər, lakin bu sxem təhlükəsiz deyil.</p>
<p><b>Praktik sxem:</b> mesaj əvvəlcə heşlənir və EMSA-PSS və ya EMSA-PKCS1-v1_5 ilə kodlanır; doğrulama eyni encoding qaydasını yoxlayır.</p>
</div>

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336; margin-top: 10px;">
<h4>⚠️ RSA imza təhlükəsizliyi</h4>
<p>Birbaşa RSA imzası (textbook RSA signature) təhlükəsiz deyil:</p>
<ul>
    <li>Multiplikativ xüsusiyyət: $s_1 \cdot s_2 = (m_1 \cdot m_2)^d \mod n$</li>
    <li>Hücumçu mövcud imzalardan yeni imzalar yarada bilər</li>
</ul>
<p><b>Həll:</b> Heş-&-imza (hash-and-sign) + padding (PKCS#1 v1.5, PSS)</p>
</div>

In [ ]:
def rsa_pss_demo():
    """
    RSA-PSS (Probabilistic Signature Scheme) imza nümayişi
    """
    if not CRYPTOGRAPHY_AVAILABLE:
        print("cryptography kitabxanası yüklü deyil")
        return
    
    print("\n" + "=" * 70)
    print("🔐 RSA-PSS İMZA SXEMİ")
    print("=" * 70)
    
    from cryptography.hazmat.primitives.asymmetric import rsa, padding
    from cryptography.hazmat.primitives import hashes
    from cryptography.exceptions import InvalidSignature
    
    # Açar generasiyası
    print("1. RSA açar cütlüyü yaradılır...")
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )
    public_key = private_key.public_key()
    
    print(f"   Açar ölçüsü: 2048-bit")
    print(f"   Açıq açar: {public_key.public_numbers().n}")
    
    # Mesaj
    message = "Bu vacib sənəddir. Heç kim dəyişdirə bilməz!".encode("utf-8")
    print(f"\n📨 Mesaj: {message}")
    
    # İmza yaratma (PSS - Probabilistic Signature Scheme)
    print("\n2. İmza yaradılır...")
    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    
    print(f"   İmza (hex): {signature.hex()}")
    print(f"   İmza uzunluğu: {len(signature)} bayt")
    
    # İmza doğrulama
    print("\n3. İmza doğrulanır...")
    try:
        public_key.verify(
            signature,
            message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        print("✅ İmza ETİBARLIDIR!")
    except InvalidSignature:
        print("❌ İmza YANLIŞDIR!")
    
    # Dəyişdirilmiş mesaj testi
    print("\n4. Dəyişdirilmiş mesaj testi...")
    modified_message = "Bu dəyişdirilmiş sənəddir!".encode("utf-8")
    try:
        public_key.verify(
            signature,
            modified_message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        print("❌ Dəyişdirilmiş mesaj doğrulandı (OLMAMALIDIR!)")
    except InvalidSignature:
        print("✅ Dəyişdirilmiş mesaj üçün imza YANLIŞDIR (düzgün!)")

rsa_pss_demo()

### 5.1 RSA-PKCS#1 v1.5 imza (PyCryptodome)

In [ ]:
def rsa_pkcs1_demo():
    """
    RSA-PKCS#1 v1.5 imza sxemi (PyCryptodome)
    """
    if not CRYPTO_AVAILABLE:
        return
    
    print("\n" + "-" * 70)
    print("📦 RSA-PKCS#1 v1.5 İMZA (PyCryptodome)")
    print("-" * 70)
    
    from Crypto.PublicKey import RSA
    from Crypto.Signature import pkcs1_15
    from Crypto.Hash import SHA256
    
    # Açar generasiyası
    key = RSA.generate(2048)
    private_key = key
    public_key = key.publickey()
    
    print(f"Açar ölçüsü: {key.size_in_bits()} bit")
    
    # Mesaj
    message = b"Bu PKCS#1 v1.5 imza testidir."
    h = SHA256.new(message)
    
    print(f"\n📨 Mesaj: {message}")
    print(f"🔢 Heş: {h.hexdigest()}")
    
    # İmza
    signature = pkcs1_15.new(private_key).sign(h)
    print(f"\n🔏 İmza (hex): {signature.hex()}")
    
    # Doğrulama
    try:
        pkcs1_15.new(public_key).verify(h, signature)
        print("✅ İmza ETİBARLIDIR!")
    except (ValueError, TypeError):
        print("❌ İmza YANLIŞDIR!")

if CRYPTO_AVAILABLE:
    rsa_pkcs1_demo()

### ✍️ Çalışma 1: RSA imza (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **RSA açar generasiyası:** 2048-bit RSA açar cütlüyü yaradın.

2. **İmza yaratma:** Öz adınızı ehtiva edən bir mesaj üçün RSA-PSS imzası yaradın.

3. **İmza doğrulama:** Yaradılan imzanı doğrulayın.

4. **PKCS#1 vs PSS:** RSA-PKCS#1 v1.5 və RSA-PSS arasında nə fərq var? Hansı daha təhlükəsizdir?

In [ ]:
# Çalışma 1 - Cavablar

print("📝 ÇALIŞMA 1 CAVABLARI")
print("=" * 80)

if CRYPTOGRAPHY_AVAILABLE:
    from cryptography.hazmat.primitives.asymmetric import rsa, padding
    from cryptography.hazmat.primitives import hashes
    
    # 1. RSA açar generasiyası
    print("\n1. RSA AÇAR GENERASİYASI:")
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )
    public_key = private_key.public_key()
    print(f"   Açar ölçüsü: 2048-bit")
    print(f"   Açıq açar (n): {public_key.public_numbers().n}")

    # 2-3. İmza yaratma və doğrulama
    print("\n2-3. İMZA YARATMA VƏ DOĞRULAMA:")
    my_name = "Səməd Vəkilov"
    message = f"Bu sənəd {my_name} tərəfindən imzalanmışdır.".encode()
    
    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    
    try:
        public_key.verify(
            signature,
            message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        print("✅ İmza uğurla yaradıldı və doğrulandı!")
    except InvalidSignature:
        print("❌ Xəta!")

# 4. PKCS#1 vs PSS
print("\n4. PKCS#1 vs PSS MÜQAYİSƏSİ:")
print("""
   RSA-PKCS#1 v1.5:
   • Daha köhnə, sadə format
   • Deterministik (eyni mesaj həmişə eyni imza)
   • Bəzi nəzəri zəifliklər aşkar edilib
   
   RSA-PSS (Probabilistic Signature Scheme):
   • Müasir, təhlükəsiz dizayn
   • Probabilistik (təsadüfi salt istifadə edir)
   • Standart təhlükəsizlik modellərində RSA fərziyyələri və heş modeli altında formal əsaslandırması var
   • PKCS#1 v2.1 və sonrakı versiyalarda standart
   
   Nəticə: RSA-PSS daha təhlükəsizdir və yeni sistemlərdə istifadə edilməlidir.
""")

## 🔑 DSA (Digital Signature Algorithm) (25 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 DSA (Digital Signature Algorithm)</h4>
<p>DSA 1991-ci ildə təklif edilmiş və 1994-cü ildə FIPS 186 ilə standartlaşdırılmış diskret-loqarifm əsaslı imza sxemidir. FIPS 186-5 yeni DSA imzalarının yaradılmasını standartın əhatəsindən çıxarıb; DSA yalnız köhnə imzaların yoxlanması üçün saxlanılır.</p>

<p><b>Açar generasiyası:</b></p>
<ul>
    <li>$p$ — L-bit sadə ədəd (L = 1024, 2048, 3072)</li>
    <li>$q$ — 160/224/256-bit sadə ədəd, $q | (p-1)$</li>
    <li>$g = h^{(p-1)/q} \mod p$, $h$ təsadüfi, $g \neq 1$</li>
    <li>$x$ — gizli açar ($1 \le x \le q-1$)</li>
    <li>$y = g^x \mod p$ — açıq açar</li>
</ul>

<p><b>İmza yaratma:</b></p>
<ol>
    <li>$k$ hər imza üçün unikal seçilir ($1 \le k \le q-1$); təhlükəsiz təsadüfi və ya standart deterministik nonce generation istifadə oluna bilər</li>
    <li>$r = (g^k \mod p) \mod q$</li>
    <li>$s = k^{-1}(H(M) + x \cdot r) \mod q$</li>
    <li>İmza = $(r, s)$</li>
</ol>

<p><b>İmza doğrulama:</b></p>
<ol>
    <li>$w = s^{-1} \mod q$</li>
    <li>$u_1 = H(M) \cdot w \mod q$</li>
    <li>$u_2 = r \cdot w \mod q$</li>
    <li>$v = (g^{u_1} \cdot y^{u_2} \mod p) \mod q$</li>
    <li>$v = r$ olmalıdır</li>
</ol>
</div>

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336; margin-top: 10px;">
<h4>⚠️ DSA təhlükəsizlik qeydləri</h4>
<ul>
    <li>$k$ hər imza üçün unikal olmalıdır. Təkrar və ya qərəzli nonce gizli açar $x$-i aça bilər.</li>
    <li>DSA-nın təhlükəsizliyi diskret loqarifm probleminin çətinliyinə əsaslanır.</li>
    <li>DSA legacy sxemidir: FIPS 186-5 yeni imza yaratmaq üçün DSA-nı təsdiqləmir; yeni sistemlərdə ECDSA, EdDSA və ya uyğun post-kvant imza profili seçilir.</li>
</ul>
</div>

In [ ]:
def dsa_demo():
    """
    DSA (Digital Signature Algorithm) nümayişi
    """
    if not CRYPTOGRAPHY_AVAILABLE:
        print("cryptography kitabxanası yüklü deyil")
        return
    
    print("\n" + "-" * 70)
    print("🔑 DSA (DIGITAL SIGNATURE ALGORITHM)")
    print("-" * 70)
    
    from cryptography.hazmat.primitives.asymmetric import dsa
    from cryptography.hazmat.primitives import hashes
    from cryptography.exceptions import InvalidSignature
    
    # Açar generasiyası
    print("1. DSA açar cütlüyü yaradılır...")
    private_key = dsa.generate_private_key(
        key_size=2048
    )
    public_key = private_key.public_key()
    
    print(f"   Açar ölçüsü: 2048-bit")
    
    # Mesaj
    message = b"Bu DSA imza testidir."
    print(f"\n📨 Mesaj: {message}")
    
    # İmza yaratma
    print("\n2. DSA imzası yaradılır...")
    signature = private_key.sign(
        message,
        hashes.SHA256()
    )
    
    print(f"   İmza (hex): {signature.hex()}")
    print(f"   İmza uzunluğu: {len(signature)} bayt")
    
    # İmza doğrulama
    print("\n3. İmza doğrulanır...")
    try:
        public_key.verify(
            signature,
            message,
            hashes.SHA256()
        )
        print("✅ İmza ETİBARLIDIR!")
    except InvalidSignature:
        print("❌ İmza YANLIŞDIR!")

if CRYPTOGRAPHY_AVAILABLE:
    dsa_demo()

### 6.1 DSA-da $k$ təkrarı problemi

In [ ]:
def dsa_k_reuse_attack_demo():
    """
    DSA-da k-nın təkrar istifadəsi probleminin nümayişi
    """
    print("\n" + "-" * 70)
    print("⚠️ DSA-da k TƏKRARI PROBLEMİ")
    print("-" * 70)
    
    print("""
    DSA-da hər imza üçün unikal k seçilməlidir. Əgər eyni k iki dəfə istifadə olunarsa:
    
    s1 = k⁻¹(H(m1) + x·r) mod q
    s2 = k⁻¹(H(m2) + x·r) mod q
    
    k = (H(m1) - H(m2)) · (s1 - s2)⁻¹ mod q
    x = (s1·k - H(m1)) · r⁻¹ mod q
    
    Nəticə: Gizli açar x bərpa oluna bilər!
    
    Eyni algebra ECDSA üçün də keçərlidir. Real nümunə: 2010-cu ildə
    Sony PlayStation 3 ECDSA implementasiyasında sabit nonce səbəbindən
    imza açarı bərpa edildi.
    """)

dsa_k_reuse_attack_demo()

### ✍️ Çalışma 2: DSA (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **DSA açar generasiyası:** 2048-bit DSA açar cütlüyü yaradın.

2. **DSA imzası:** Bir mesaj üçün DSA imzası yaradın və doğrulayın.

3. **$k$ təkrarı:** DSA-da $k$ parametrinin təkrar istifadəsi niyə təhlükəlidir? Riyazi izah verin.

4. **RSA vs DSA:** RSA və DSA arasında 3 əsas fərqi sadalayın.

In [ ]:
# Çalışma 2 - Cavablar

print("📝 ÇALIŞMA 2 CAVABLARI")
print("=" * 80)

if CRYPTOGRAPHY_AVAILABLE:
    from cryptography.hazmat.primitives.asymmetric import dsa
    from cryptography.hazmat.primitives import hashes
    
    # 1-2. DSA açar generasiyası və imza
    print("\n1-2. DSA AÇAR GENERASİYASI VƏ İMZA:")
    private_key = dsa.generate_private_key(key_size=2048)
    public_key = private_key.public_key()
    
    message = b"DSA test mesaji"
    signature = private_key.sign(message, hashes.SHA256())
    
    try:
        public_key.verify(signature, message, hashes.SHA256())
        print("✅ DSA imzası uğurla yaradıldı və doğrulandı!")
    except InvalidSignature:
        print("❌ Xəta!")

# 3. k təkrarı problemi
print("\n3. k TƏKRARI PROBLEMİ:")
print("""
   DSA-da eyni k iki dəfə istifadə edilərsə:
   
   s1 = k⁻¹(H(m1) + x·r) mod q
   s2 = k⁻¹(H(m2) + x·r) mod q
   
   s1 - s2 = k⁻¹(H(m1) - H(m2)) mod q
   k = (H(m1) - H(m2)) · (s1 - s2)⁻¹ mod q
   
   k tapıldıqdan sonra: x = (s1·k - H(m1)) · r⁻¹ mod q
   
   Nəticə: Gizli açar tamamilə bərpa olunur!
""")

# 4. RSA vs DSA
print("\n4. RSA vs DSA FƏRQLƏRİ:")
print("""
   1) Riyazi əsas:
      • RSA: Faktorizasiya problemi
      • DSA: Diskret loqarifm problemi
   
   2) İmza ölçüsü:
      • RSA: Təxminən 256 bayt (2048-bit)
      • DSA: ölçü q-dan və kodlamadan asılıdır; 256-bit q üçün raw r||s 64 baytdır, DER isə dəyişkəndir
   
   3) Təsadüfilik tələbi:
      • RSA-PKCS#1 v1.5 deterministikdir; RSA-PSS təsadüfi salt istifadə edir
      • DSA hər imza üçün unikal nonce tələb edir; təhlükəsiz deterministik generation mümkündür
""")

## ⚡ ECDSA (Elliptic Curve DSA) (20 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>⚡ ECDSA (Elliptic Curve Digital Signature Algorithm)</h4>
<p>ECDSA, DSA-nın elliptik əyrilər üzərində tətbiqidir. Daha kiçik açar ölçüləri ilə eyni təhlükəsizlik səviyyəsini təmin edir.</p>

<p><b>Açar generasiyası:</b></p>
<ul>
    <li>Elliptik əyri $E$ və baza nöqtəsi $G$ seçilir</li>
    <li>$n$ — $G$-nin sırası (order)</li>
    <li>$d$ — gizli açar ($1 \le d \le n-1$)</li>
    <li>$Q = d \cdot G$ — açıq açar</li>
</ul>

<p><b>İmza yaratma:</b></p>
<ol>
    <li>$k$ — hər imza üçün unikal və proqnozlaşdırılması mümkün olmayan, yaxud standart deterministik üsulla yaradılan qiymət ($1 \le k \le n-1$)</li>
    <li>$R = k \cdot G = (x_R, y_R)$</li>
    <li>$r = x_R \mod n$</li>
    <li>$s = k^{-1}(H(M) + d \cdot r) \mod n$</li>
    <li>İmza = $(r, s)$</li>
</ol>
</div>

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50; margin-top: 10px;">
<h4>📌 Açar ölçüləri müqayisəsi</h4>
<table>
<tr><th>Təhlükəsizlik</th><th>RSA ölçüsü</th><th>ECDSA ölçüsü</th></tr>
<tr><td>80-bit</td><td>1024-bit</td><td>160-bit</td></tr>
<tr><td>112-bit</td><td>2048-bit</td><td>224-bit</td></tr>
<tr><td>128-bit</td><td>3072-bit</td><td>256-bit</td></tr>
<tr><td>192-bit</td><td>7680-bit</td><td>384-bit</td></tr>
<tr><td>256-bit</td><td>15360-bit</td><td>521-bit</td></tr>
</table>
</div>

In [ ]:
def ecdsa_demo():
    """
    ECDSA (Elliptic Curve DSA) nümayişi
    """
    if not CRYPTOGRAPHY_AVAILABLE:
        return
    
    print("\n" + "-" * 70)
    print("⚡ ECDSA (ELLIPTIC CURVE DSA)")
    print("-" * 70)
    
    from cryptography.hazmat.primitives.asymmetric import ec
    from cryptography.hazmat.primitives import hashes
    from cryptography.exceptions import InvalidSignature
    
    # Müxtəlif əyri seçimləri
    curves = [
        (ec.SECP256R1, "secp256r1 (256-bit)"),
        (ec.SECP384R1, "secp384r1 (384-bit)"),
        (ec.SECP521R1, "secp521r1 (521-bit)")
    ]
    
    for curve_class, curve_name in curves:
        print(f"\n📌 {curve_name}:")
        
        # Açar generasiyası
        private_key = ec.generate_private_key(curve_class())
        public_key = private_key.public_key()
        
        # Mesaj
        message = b"ECDSA test mesaji"
        
        # İmza
        signature = private_key.sign(
            message,
            ec.ECDSA(hashes.SHA256())
        )
        
        print(f"   İmza ölçüsü: {len(signature)} bayt")
        
        # Doğrulama
        try:
            public_key.verify(
                signature,
                message,
                ec.ECDSA(hashes.SHA256())
            )
            print(f"   ✅ İmza etibarlıdır")
        except InvalidSignature:
            print(f"   ❌ İmza yanlışdır")

if CRYPTOGRAPHY_AVAILABLE:
    ecdsa_demo()

### ✍️ Çalışma 3: ECDSA (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Fərqli əyrilər:** secp256r1, secp384r1, secp521r1 əyriləri üçün ECDSA imzaları yaradın. İmza ölçülərini müqayisə edin.

2. **RSA vs ECDSA:** 3072-bit RSA ilə 256-bit ECDSA-nın təhlükəsizlik səviyyəsini müqayisə edin.

3. **Performans:** ECDSA-nın RSA-dan üstünlükləri nələrdir?

4. **Bitcoin:** Bitcoin hansı əyrini istifadə edir? Niyə?

In [ ]:
# Çalışma 3 - Cavablar

print("📝 ÇALIŞMA 3 CAVABLARI")
print("=" * 80)

if CRYPTOGRAPHY_AVAILABLE:
    from cryptography.hazmat.primitives.asymmetric import ec
    from cryptography.hazmat.primitives import hashes
    
    # 1. Fərqli əyrilər
    print("\n1. FƏRQLİ ƏYRİLƏR:")
    curves = [
        (ec.SECP256R1, "secp256r1", 256),
        (ec.SECP384R1, "secp384r1", 384),
        (ec.SECP521R1, "secp521r1", 521)
    ]
    
    message = b"Test mesaji"
    for curve_class, name, bits in curves:
        private_key = ec.generate_private_key(curve_class())
        signature = private_key.sign(message, ec.ECDSA(hashes.SHA256()))
        print(f"   {name}: imza ölçüsü = {len(signature)} bayt")

# 2. RSA vs ECDSA təhlükəsizlik
print("\n2. RSA vs ECDSA TƏHLÜKƏSİZLİK:")
print("""
   3072-bit RSA ≈ P-256 ECDSA (təxminən 128-bit klassik təhlükəsizlik)
   
   NIST-in müqayisəli qiymətləndirməsində bu parametrlər eyni təhlükəsizlik səviyyəsinə uyğundur.
   ECC üçün ümumi diskret loqarifm hücumları təxminən 2^128 əməliyyat tələb edir.
""")

# 3. ECDSA üstünlükləri
print("\n3. ECDSA ÜSTÜNLÜKLƏRİ:")
print("""
   • Müqayisə olunan təhlükəsizlikdə açarlar xeyli kiçikdir (P-256 vs RSA-3072)
   • İmza adətən RSA-dan kiçikdir; ECDSA üçün DER ölçüsü dəyişkəndir, xam (r || s) P-256 formatı 64 baytdır
   • Sürət və enerji sərfi əməliyyatdan, kitabxanadan və platformadan asılıdır; etibarlı nəticə üçün ölçmək lazımdır
   • Açar və nonce generasiyası etibarlı təsadüfilik və ya standart deterministik nonce üsulu tələb edir
""")

# 4. Bitcoin
print("\n4. BİTCOİN:")
print("""
   Bitcoin secp256k1 əyrisini istifadə edir:
   • y² = x³ + 7 (a=0, b=7)
   • Səmərəli implementasiyalar üçün secp256k1 endomorfizmindən istifadə oluna bilər
   • Təxminən 128-bit klassik təhlükəsizlik
   
   Seçim səbəbləri:
   • Sadə parametrlər (a=0, b=7)
   • Layihənin seçdiyi ekosistemlə uyğunluq
   • Effektiv implementasiya
""")

## ⚡ Ed25519 - Müasir imza sxemi (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>⚡ Ed25519</h4>
<p>Ed25519, Daniel J. Bernstein tərəfindən hazırlanmış müasir imza sxemidir. EdDSA (Edwards-curve Digital Signature Algorithm) ailəsinə aiddir.</p>

<p><b>Xüsusiyyətlər:</b></p>
<ul>
    <li>✅ Təxminən 128-bit klassik təhlükəsizlik</li>
    <li>✅ Performansı əməliyyatdan, kitabxanadan və platformadan asılı olaraq ölçülür</li>
    <li>✅ Deterministik nonce generasiyası təsadüfi nonce səhvləri riskini azaldır</li>
    <li>✅ Xam açıq açar və gizli seed 32 bayt, imza 64 baytdır</li>
    <li>✅ Sxem sabit-vaxt implementasiyasını asanlaşdırır; konkret kitabxana ayrıca yoxlanmalıdır</li>
</ul>

<p><b>İstifadə nümunələri:</b> OpenSSH və Tor</p>
</div>

In [ ]:
def ed25519_demo():
    """
    Ed25519 imza sxeminin nümayişi
    """
    if not CRYPTOGRAPHY_AVAILABLE:
        return
    
    print("\n" + "-" * 70)
    print("⚡ Ed25519 (MODERN SIGNATURE SCHEME)")
    print("-" * 70)
    
    from cryptography.hazmat.primitives.asymmetric import ed25519
    from cryptography.hazmat.primitives import serialization
    from cryptography.exceptions import InvalidSignature
    
    # Açar generasiyası
    print("1. Ed25519 açar cütlüyü yaradılır...")
    private_key = ed25519.Ed25519PrivateKey.generate()
    public_key = private_key.public_key()
    
    private_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PrivateFormat.Raw,
        encryption_algorithm=serialization.NoEncryption()
    )
    public_bytes = public_key.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw
    )
    
    print(f"   Gizli açar ölçüsü: {len(private_bytes)} bayt")
    print(f"   Açıq açar ölçüsü: {len(public_bytes)} bayt")
    
    # Mesaj
    message = b"Bu Ed25519 imza testidir."
    print(f"\n📨 Mesaj: {message}")
    
    # İmza
    print("\n2. Ed25519 imzası yaradılır...")
    signature = private_key.sign(message)
    print(f"   İmza ölçüsü: {len(signature)} bayt")
    print(f"   İmza (hex): {signature.hex()}")
    
    # Doğrulama
    print("\n3. İmza doğrulanır...")
    try:
        public_key.verify(signature, message)
        print("✅ İmza ETİBARLIDIR!")
    except InvalidSignature:
        print("❌ İmza YANLIŞDIR!")

if CRYPTOGRAPHY_AVAILABLE:
    ed25519_demo()

### ✍️ Çalışma 4: Ed25519 (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Ed25519 imzası:** Bir mesaj üçün Ed25519 imzası yaradın.

2. **Açar ölçüləri:** Ed25519 açarlarının ölçülərini RSA (2048-bit) ilə müqayisə edin.

3. **Ed25519 üstünlükləri:** Ed25519-un ECDSA-dan üstünlükləri nələrdir?

4. **Praktik tətbiqlər:** Hansı protokollar Ed25519 istifadə edir?

In [ ]:
# Çalışma 4 - Cavablar

print("📝 ÇALIŞMA 4 CAVABLARI")
print("=" * 80)

if CRYPTOGRAPHY_AVAILABLE:
    from cryptography.hazmat.primitives.asymmetric import ed25519
    
    # 1. Ed25519 imzası
    print("\n1. Ed25519 İMZASI:")
    private_key = ed25519.Ed25519PrivateKey.generate()
    public_key = private_key.public_key()
    
    message = b"Test mesaji"
    signature = private_key.sign(message)
    
    try:
        public_key.verify(signature, message)
        print("✅ Ed25519 imzası uğurla yaradıldı və doğrulandı!")
    except InvalidSignature:
        print("❌ Xəta!")

# 2. Açar ölçüləri
print("\n2. AÇAR ÖLÇÜLƏRİ MÜQAYİSƏSİ:")
print("""
   Ed25519:
   • Xam gizli seed: 32 bayt
   • Xam açıq açar: 32 bayt
   • İmza: 64 bayt
   
   RSA-2048:
   • Modul və imza: 256 bayt
   • Gizli və açıq açar fayllarının ölçüsü DER/PEM və formatdan asılıdır
   
   Buna görə yalnız eyni kodlaşdırma formatındakı konkret obyektləri müqayisə etmək düzgündür.
""")

# 3. Ed25519 üstünlükləri
print("\n3. Ed25519 ÜSTÜNLÜKLƏRİ:")
print("""
   • Standart deterministik nonce generasiyası təsadüfi nonce səhvləri riskini azaldır
   • Sabit 32-bayt açıq açar və 64-bayt imza formatı verir
   • Əyri və parametrlər əvvəlcədən müəyyən edildiyi üçün seçim səhvləri azalır
   • Sürət üstünlüyü universal deyil; imzalama və doğrulama ayrıca benchmark edilməlidir
   • Yan-kanal davamlılığı konkret implementasiyadan asılıdır
""")

# 4. Praktik tətbiqlər
print("\n4. PRAKTİK TƏTBİQLƏR:")
print("""
   Ed25519 istifadə edən protokollar:
   • OpenSSH (ssh-keygen -t ed25519)
   • Tor (The Onion Router)
   • RFC 8032-yə uyğun kitabxana və tətbiqlər

   Qeyd: WireGuard açar razılaşması üçün Curve25519/X25519 istifadə edir; protokolun ayrıca rəqəmsal imza qatı yoxdur.
""")

## 📊 Rəqəmsal imzaların müqayisəsi (10 dəq)

In [ ]:
def signature_comparison():
    """
    Rəqəmsal imza sxemlərinin müqayisəsi
    """
    print("\n" + "-" * 90)
    print("📊 RƏQƏMSAL İMZA SXEMLƏRİNİN MÜQAYİSƏSİ")
    print("-" * 90)
    
    rows = [
        ("RSA-2048", "~112 bit", "kodlaşdırmadan asılıdır", "256 bayt", "PSS-də salt", "uyğunluq"),
        ("DSA-2048", "q-dan asılıdır", "kodlaşdırmadan asılıdır", "DER dəyişkən", "unikal/deterministik", "yalnız köhnə imzanı yoxlama"),
        ("ECDSA P-256", "~128 bit", "33/65 bayt nöqtə", "DER dəyişkən; xam 64", "unikal/deterministik", "TLS, Bitcoin*"),
        ("Ed25519", "~128 bit", "xam 32 bayt", "64 bayt", "deterministik", "OpenSSH, Tor"),
    ]
    headers = ("Alqoritm", "Təhlükəsizlik", "Açıq açar", "İmza", "Nonce/salt", "Status/nümunə")
    print(" | ".join(headers))
    print("-" * 120)
    for row in rows:
        print(" | ".join(row))
    print("\n* Bitcoin secp256k1 istifadə edir; P-256 sətri yalnız ölçü sinfini göstərir.")

signature_comparison()

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50;">
<h4>📌 Nə vaxt hansı imza sxemini seçməli?</h4>
<ul>
    <li><b>Köhnə sistemlərlə uyğunluq:</b> RSA-PKCS#1 v1.5</li>
    <li><b>TLS və sertifikatlar:</b> protokol profili və müştəri dəstəyi imkan verirsə RSA-PSS, ECDSA və ya Ed25519</li>
    <li><b>Kriptovalyutalar:</b> ECDSA (secp256k1)</li>
    <li><b>Yeni sistemlər:</b> ekosistem dəstəkləyirsə Ed25519; əks halda uyğun standart profil</li>
    <li><b>Parametr seçimi:</b> tələb olunan təhlükəsizlik səviyyəsini və standart profilini əsas götürün; eyni siyahıdakı açar ölçüləri həmişə eyni gücdə deyil</li>
</ul>
</div>

## 🖥️ İnteqrasiya edilmiş tətbiq (15 dəq)

In [ ]:
def signature_lab_menu():
    """
    Rəqəmsal imza laboratoriyası - əlavə AS-23b
    """
    while True:
        print("\n" + "=" * 70)
        print("🔏 RƏQƏMSAL İMZA LABORATORİYASI - Əlavə AS-23b")
        print("=" * 70)
        print("1. 🔐 RSA-PSS imza (cryptography)")
        print("2. 📦 RSA-PKCS#1 v1.5 imza (PyCryptodome)")
        print("3. 🔑 DSA imza")
        print("4. ⚡ ECDSA imza")
        print("5. ⚡ Ed25519 imza")
        print("6. 📊 İmza müqayisəsi")
        print("7. ⚠️ DSA k təkrarı problemi")
        print("0. ❌ Çıxış")
        print("=" * 70)
        
        choice = input("📌 Seçiminiz: ")
        
        if choice == '1':
            rsa_pss_demo()
        elif choice == '2':
            rsa_pkcs1_demo()
        elif choice == '3':
            dsa_demo()
        elif choice == '4':
            ecdsa_demo()
        elif choice == '5':
            ed25519_demo()
        elif choice == '6':
            signature_comparison()
        elif choice == '7':
            dsa_k_reuse_attack_demo()
        elif choice == '0':
            print("👋 Proqram bitdi. Sağ olun!")
            break
        else:
            print("❌ Yanlış seçim və ya kitabxana yüklü deyil")

# Proqramı işə sal (istəyə bağlı)
# signature_lab_menu()

## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: Rəqəmsal imza paketi (3 bal)

Aşağıdakı funksiyaları özündə birləşdirən Python paketi yazın:

```
signature_package/
├── __init__.py
├── rsa_signature.py    # RSA-PSS və RSA-PKCS#1 imza nümayişi
├── dsa_signature.py    # DSA imza, k təkrarı problemi
├── ecdsa_signature.py  # ECDSA müxtəlif əyrilərlə
├── ed25519_signature.py # Ed25519 imza
├── comparison.py       # İmza sxemlərinin müqayisəsi
└── main.py             # İnteraktiv menyu
```

**Tələblər:**
* Hər bir funksiya üçün docstring yazın
* Səhv hallarını idarə edin (try-except)
* Kod təmiz və oxunaqlı olmalıdır
* Hər modul üçün test nümunələri əlavə edin

### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

Aşağıdakı məsələləri həll edin:

1. **RSA imza performansı:** 2048, 3072 və 4096-bit RSA açarları üçün imza yaratma və doğrulama vaxtlarını təkrarlanan ölçmələrlə müqayisə edin. RSA-1024-ün müasir istifadə üçün təhlükəsiz sayılmadığını ayrıca qeyd edin.

2. **ECDSA vs Ed25519:** ECDSA (secp256r1) və Ed25519-un performansını müqayisə edən proqram yazın.

3. **Açar faylları:** RSA, DSA, ECDSA və Ed25519 açarlarını PEM formatında fayla yazın və oxuyun.

4. **Sertifikat imzalama:** Sadə bir sertifikat strukturu yaradın və onu RSA ilə imzalayın.

### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

Araşdırma aparın və aşağıdakı suallara cavab tapın. Cavablarınızı 1-2 səhifəlik hesabat şəklində təqdim edin:

1. **RSA-nın ixtiraçıları:** Ron Rivest, Adi Shamir, Leonard Adleman kimdir? RSA-nın yaranma hekayəsi nədir?

2. **DSA və NSA:** DSA-nın yaradılmasında NSA-nın rolu nə olub? Niyə DSA patentlə bağlı problem yaratmır?

3. **Elliptik əyrilər:** NIST tərəfindən standartlaşdırılan əyrilər (P-256, P-384, P-521) ilə Bitcoin-in secp256k1 əyrisi arasında nə fərq var?

4. **Ed25519 və DJB:** Daniel J. Bernstein-ın kriptoqrafiyaya töhfələri nələrdir? Ed25519-un digər imza sxemlərindən üstünlükləri nələrdir?

**Format tələbləri:**
* PDF formatında təqdim edin
* Ən azı 5 qaynaq göstərin (kitab, məqalə, veb səhifə)
* Öz sözlərinizlə yazın (copy-paste yox)
* Mümkünsə, sxemlər və ya diaqramlar əlavə edin

## 📌 Yekun və müzakirə sualları

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; border-left: 5px solid #2c3e50;">
<h3>📋 Xülasə</h3>
<p>Bu məşğələdə öyrəndiklərimiz:</p>
<ul>
    <li>✅ <b>RSA imza:</b> Faktorizasiya probleminə əsaslanır. Yeni dizaynlarda RSA-PSS üstün tutulur; PKCS#1 v1.5 əsasən uyğunluq üçündür.</li>
    <li>✅ <b>DSA:</b> Diskret loqarifm probleminə əsaslanır. Unikal və proqnozlaşdırılması mümkün olmayan, yaxud standart deterministik nonce tələb edir; FIPS 186-5 yeni DSA imzalarının yaradılmasını təsdiqləmir.</li>
    <li>✅ <b>ECDSA:</b> Elliptik əyrilər üzərində DSA. Daha kiçik açarlarla eyni təhlükəsizlik.</li>
    <li>✅ <b>Ed25519:</b> Müasir, deterministik nonce generasiyalı imza sxemidir; performans və yan-kanal davamlılığı konkret implementasiyadan asılıdır.</li>
</ul>
<p><i>Rəqəmsal imza açıq yoxlama imkanı verir. İnkar edilməzlik isə açarın kimliyə etibarlı bağlanması, açarın qorunması, protokol sübutları və hüquqi kontekstdən asılıdır.</i></p>
</div>

### 💭 Müzakirə sualları

1. Bu məşğələdə ən maraqlı tapdığınız nə oldu?
2. Rəqəmsal imza ilə MAK arasında ən mühüm fərq nədir?
3. DSA-da k təkrarı niyə təhlükəlidir? Sony PlayStation 3 hadisəsi haqqında nə bilirsiniz?
4. Ed25519-un ECDSA-dan üstünlükləri nələrdir?
5. RSA-PSS nə üçün RSA-PKCS#1 v1.5-dən daha təhlükəsizdir?
6. Bitcoin niyə secp256k1 əyrisini seçib?
7. Kvant kompüterlər mövcud imza sxemlərinə təhlükə yaradırmı? (Növbəti məşğələnin mövzusu)

## 📚 Əlavə resurslar

* 📘 **RFC 8017 (PKCS#1):** [https://www.rfc-editor.org/rfc/rfc8017](https://www.rfc-editor.org/rfc/rfc8017)
* 📙 **FIPS 186-5 (DSA, ECDSA):** [https://csrc.nist.gov/publications/detail/fips/186/5/final](https://csrc.nist.gov/publications/detail/fips/186/5/final)
* 📗 **Ed25519 (RFC 8032):** [https://www.rfc-editor.org/rfc/rfc8032](https://www.rfc-editor.org/rfc/rfc8032)
* 📕 **SEC 2 (Elliptic Curves):** [https://www.secg.org/sec2-v2.pdf](https://www.secg.org/sec2-v2.pdf)
* 📘 **cryptography docs:** [https://cryptography.io/](https://cryptography.io/)
* 📙 **PyCryptodome docs:** [https://pycryptodome.readthedocs.io/](https://pycryptodome.readthedocs.io/)
* 📗 **Daniel J. Bernstein-in səhifəsi:** [https://cr.yp.to/djb.html](https://cr.yp.to/djb.html)
* 📘 **Sony PS3 incident:** [https://events.ccc.de/congress/2010/Fahrplan/events/4087.en.html](https://events.ccc.de/congress/2010/Fahrplan/events/4087.en.html)

---

<div style="background-color: #f0f0f0; padding: 20px; border-radius: 10px; text-align: center;">
<h2>✅ Əlavə AS-23b tamamlandı!</h2>
<p>Bütün kodları və tapşırıq cavablarını növbəti məşğələyə qədər təqdim edin.</p>
<p><em>Kodlar aydın şərhlərlə yazılmalı və asan oxunmalıdır. Hər bir funksiyanın nə etdiyini izah edən şərhlər əlavə edin.</em></p>
<p style="font-size: 1.2em; margin-top: 15px;">🔏 <b>Rəqəmsal imzalar - rəqəmsal dünyada kimliyinizi təsdiqləyin!</b></p>
</div>